## Overview of specific cases for making E.coli models auxotrophs to the wanted substrate

In [3]:
import requests
import reframed
import pandas as pd
import warnings

In [4]:
import sys

sys.path.append("../functions/")
from EGC import EGC_identifier

**Load model**

In [5]:
model_ecoli = reframed.load_cbmodel("../models/Guillen2021/iML1515.xml")

**Load media**

*For E. coli mutants (LB medium to see auxotrophies)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="LB"].compound.values)
compounds.append("btn") # biot members are auxotrophic to biotin which was not in LB-medium. 
compounds.append("mmet") # metE and metH double mutant has been shown to be dependent on mmet (S-Methylmethionine). metH is missing in the model!
compounds.append("26dap__M") # meso-DAP. Essential part of cell wall in gram-negative bacteria...
env_ecoli = reframed.Environment.from_compounds(compounds)

**Load other data**

In [7]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t",index_col="id")

In [8]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

### Special case - panZ is not in model

NB! panZ (also known as yhhK) are missing names in the model. 

In [9]:
set(relative_abundance_ecoli.index)-set(gene_id2name.name.values)

{'panZ'}

In [10]:
"yhhK" in gene_id2name.name.values

False

In [11]:
# b3459 is the gene id for panZ and it is not in the model
"G_b3459" in gene_id2name.index

False

### Special case - cysW protein and cysteine auxotrophy

Removal of cysW ws not enough to make a cysteine auxotroph - we also had to remove cysZ and modABC

In [12]:

env_ecoli_no_cys = env_ecoli.copy()
env_ecoli_no_cys.pop("R_EX_cys__L_e")

(-10.0, inf)

*cysW mutant without any other maipulations*

In [13]:
mutant= "cysW"
model = model_ecoli.copy()

reframed.hard_knockout(model,["G_b2423"],inplace=True) 

env_ecoli_no_cys.apply(model,inplace=True,exclusive=True,warning=False)
    
sol = reframed.FBA(model)
print(f"Without cysteine: {mutant}: {sol.fobj}")


env_ecoli.apply(model,inplace=True,exclusive=True,warning=False)
    
sol = reframed.FBA(model)
print(f"With cysteine {mutant}: {sol.fobj}")


Without cysteine: cysW: 3.422360625716655
With cysteine cysW: 3.432440288449043


*cysW mutant with removal of modABC and cysZ*

In [14]:
mutant= "cysW"
model = model_ecoli.copy()

model.reactions.R_SULabcpp.gpr.proteins.pop(1) # Remove molybdate genes
model.update()

reframed.hard_knockout(model,["G_b2423","G_b2413"],inplace=True) 
#G_b2413 is an alternative cys importer, that likely is less active

env_ecoli_no_cys.apply(model,inplace=True,exclusive=True,warning=False)
    
sol = reframed.FBA(model)
print(f"Without cysteine: {mutant}: {sol.fobj}")


env_ecoli.apply(model,inplace=True,exclusive=True,warning=False)
    
sol = reframed.FBA(model)
print(f"With cysteine {mutant}: {sol.fobj}")


Without cysteine: cysW: 0.0
With cysteine cysW: 3.4303524420095473


## Overview of specific cases for ensuring production

*Check what models for 7 dominating members can or cannot produce needed compounds from medium*

In [15]:
extra_compounds = ['R_EX_cys__L_e',
 'R_EX_pro__L_e',
 'R_EX_glu__L_e',
 'R_EX_pnto__R_e',
 'R_EX_btn_e']

In [16]:
can_produce = {}
for rxn in extra_compounds:
    objective = {"R_BIOMASS_Ec_iML1515_core":0, 
                 rxn:1}

    env_ecoli.apply(model_ecoli,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model_ecoli,objective=objective)
    can_produce[rxn]=sol.fobj
pd.Series(can_produce)

/Users/idunmariaburgos/opt/anaconda3/envs/rcell_rec/lib/python3.8/site-packages/reframed/solvers/cplex_solver.py:252: UserWarning: Objective variable not previously declared: R_BIOMASS_Ec_iML1515_core
  warn(f"Objective variable not previously declared: {r_id}")


R_EX_cys__L_e      10.000000
R_EX_pro__L_e     122.589333
R_EX_glu__L_e      92.175540
R_EX_pnto__R_e      0.000000
R_EX_btn_e          0.000000
dtype: float64

### Biotin production

In [17]:
for rxn in model_ecoli.get_metabolite_reactions("M_btn_e"):
    print(f"{model_ecoli.reactions[rxn]}:{model_ecoli.reactions[rxn].gpr}")

R_EX_btn_e: M_btn_e <->  [-10.0, inf]:None
R_BTNtex: M_btn_e <-> M_btn_p:(G_b2215 or G_b1377 or G_b0241 or G_b0929)


In [18]:
for rxn in model_ecoli.get_metabolite_reactions("M_btn_p"):
    print(f"{model_ecoli.reactions[rxn]}:{model_ecoli.reactions[rxn].gpr}")

R_BTNt2ipp: M_btn_p + M_h_p --> M_btn_c + M_h_c:None
R_BTNtex: M_btn_e <-> M_btn_p:(G_b2215 or G_b1377 or G_b0241 or G_b0929)


In [19]:
gene_id2name[gene_id2name.index.isin(model_ecoli.reactions.R_BTNtex.gpr.get_genes())]

,name
id,
G_b0241,phoE
G_b0929,ompF
G_b1377,ompN
G_b2215,ompC


In [20]:
gene_id2name[gene_id2name.name.isin(["YigM","bioY"])]

,name
id,


*Symport*

In [21]:
EGC_identifier(model_ecoli)

There are NO energy producing cycles in the model


{}

In [22]:
model_test = model_ecoli.copy()

model_test.reactions.R_BTNt2ipp.reversible=True
model_test.reactions.R_BTNt2ipp.lb = -1000

EGC_identifier(model_test)

There are NO energy producing cycles in the model


{}

In [23]:
can_produce = {}
for rxn in extra_compounds:
    objective = {"R_BIOMASS_Ec_iML1515_core_75p37M":0, 
                 rxn:1}

    env_ecoli.apply(model_test,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model_test,objective=objective)
    can_produce[rxn]=sol.fobj
pd.Series(can_produce)

R_EX_cys__L_e      10.000000
R_EX_pro__L_e     122.589333
R_EX_glu__L_e      92.175540
R_EX_pnto__R_e      0.000000
R_EX_btn_e         12.843779
dtype: float64

*Diffusion*

In [24]:
model_test = model_ecoli.copy()

model_test.add_reaction_from_str("R_BTNtex2: M_btn_c --> M_btn_e")
EGC_identifier(model_test)

There are NO energy producing cycles in the model


{}

In [25]:
can_produce = {}
for rxn in extra_compounds:
    objective = {"R_BIOMASS_Ec_iML1515_core_75p37M":0, 
                 rxn:1}

    env_ecoli.apply(model_test,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model_test,objective=objective)
    can_produce[rxn]=sol.fobj
pd.Series(can_produce)

R_EX_cys__L_e      10.000000
R_EX_pro__L_e     122.589333
R_EX_glu__L_e      92.175540
R_EX_pnto__R_e      0.000000
R_EX_btn_e         12.707866
dtype: float64

### Panthotenate production

In [26]:
for rxn in model_ecoli.get_metabolite_reactions("M_pnto__R_e"):
    print(f"{model_ecoli.reactions[rxn]}:{model_ecoli.reactions[rxn].gpr}")

R_PNTOtex: M_pnto__R_e <-> M_pnto__R_p:(G_b0241 or G_b0929 or G_b1377 or G_b2215)
R_EX_pnto__R_e: M_pnto__R_e <->  [-10.0, inf]:None


In [27]:
for rxn in model_ecoli.get_metabolite_reactions("M_pnto__R_p"):
    print(f"{model_ecoli.reactions[rxn]}:{model_ecoli.reactions[rxn].gpr}")

R_PNTOtex: M_pnto__R_e <-> M_pnto__R_p:(G_b0241 or G_b0929 or G_b1377 or G_b2215)
R_PNTOt4pp: M_na1_p + M_pnto__R_p --> M_na1_c + M_pnto__R_c:G_b3258


*Diffusion*

In [28]:
model_test = model_ecoli.copy()

model_test.add_reaction_from_str("R_PNTOtex2: M_pnto__R_c --> M_pnto__R_e")
EGC_identifier(model_test)

There are NO energy producing cycles in the model


{}

In [29]:
can_produce = {}
for rxn in extra_compounds:
    objective = {"R_BIOMASS_Ec_iML1515_core_75p37M":0, 
                 rxn:1}

    env_ecoli.apply(model_test,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model_test,objective=objective)
    can_produce[rxn]=sol.fobj
pd.Series(can_produce)

R_EX_cys__L_e      10.000000
R_EX_pro__L_e     122.589333
R_EX_glu__L_e      92.175540
R_EX_pnto__R_e     67.224889
R_EX_btn_e          0.000000
dtype: float64

### Both compounds

*Diffusion*

In [30]:
model_test = model_ecoli.copy()

model_test.add_reaction_from_str("R_PNTOtex2: M_pnto__R_c --> M_pnto__R_e")
model_test.add_reaction_from_str("R_BTNtex2: M_btn_c --> M_btn_e")
EGC_identifier(model_test)

There are NO energy producing cycles in the model


{}

In [31]:
can_produce = {}
for rxn in extra_compounds:
    objective = {"R_BIOMASS_Ec_iML1515_core_75p37M":0, 
                 rxn:1}

    env_ecoli.apply(model_test,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model_test,objective=objective)
    can_produce[rxn]=sol.fobj
pd.Series(can_produce)

R_EX_cys__L_e      10.000000
R_EX_pro__L_e     122.589333
R_EX_glu__L_e      92.175540
R_EX_pnto__R_e     67.224889
R_EX_btn_e         12.707866
dtype: float64